# 03 — Power Analysis

This notebook is dedicated entirely to **experiment planning**: before we spend a single day
running a test, we need to know how many customers we need, and what effect sizes we could
realistically detect.

## Setting the Minimum Detectable Effect (MDE)

From Notebook 1: current **revenue/customer ≈ $120** (mean).

The business tells us:

> "We only care about improvements of at least 5%."

So **MDE = 5%** of $120 = **$6 / customer**. Anything smaller isn't worth the operational cost of
ever raising prices, so we don't need to be able to detect it.

We now compute:
- **alpha** (significance level) — probability of a false positive we're willing to accept
- **power** — probability of detecting a true effect of size MDE, if it exists
- **sample size** — customers needed per group
- **experiment duration** — how long that takes to accumulate, given daily traffic


In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_processing import BusinessConfig, generate_synthetic_customers, compute_business_metrics
from stats_tools import plan_experiment, required_sample_size, power_given_n, mde_given_n, cohens_d

plt.style.use("seaborn-v0_8-whitegrid")

customers = generate_synthetic_customers(BusinessConfig(seed=42))
metrics = compute_business_metrics(customers)
baseline_mean = metrics["revenue_per_customer"]["mean"]
baseline_std = metrics["revenue_per_customer"]["std"]
print(f"Baseline mean revenue/customer: ${baseline_mean:,.2f}")
print(f"Baseline std revenue/customer:  ${baseline_std:,.2f}")


## 1. Core power analysis: alpha, power, MDE -> sample size

In [ ]:
ALPHA = 0.05
POWER = 0.80
MDE_RELATIVE = 0.05   # business says: we only care about >= 5% improvements

result = plan_experiment(
    baseline_mean=baseline_mean,
    baseline_std=baseline_std,
    mde_relative=MDE_RELATIVE,
    alpha=ALPHA,
    power=POWER,
)
result


In [ ]:
print(f"MDE (absolute):           ${result.mde_absolute:,.2f} per customer")
print(f"Effect size (Cohen's d):  {result.effect_size:.4f}  <- this is TINY, revenue is noisy")
print(f"Required n per group:     {result.required_n_per_group:,}")
print(f"Required total sample:    {result.required_n_per_group * 2:,}")


### Why is the required sample size so large?

Revenue per customer is a **high-variance, often zero-inflated** metric (many customers spend
nothing in a given period). A 5% relative lift on a noisy metric translates into a *very small*
standardized effect size (Cohen's d), and small effect sizes need large samples to detect reliably.

This is a genuinely important finding to bring back to the business **before** running anything:
if daily traffic can't support this sample size in a reasonable time window, we have three options:
1. Accept a longer experiment duration,
2. Raise the MDE threshold (only care about bigger effects), or
3. Reduce variance in the metric (e.g. by using CUPED / pre-experiment covariates — noted as a
   future extension in the README).

## 2. Experiment duration

Given a daily volume of new/active customers entering the experiment, how many days do we need?

In [ ]:
DAILY_ELIGIBLE_CUSTOMERS = 1500   # customers/day that can be randomized into the test

total_needed = result.required_n_per_group * 2
duration_days = np.ceil(total_needed / DAILY_ELIGIBLE_CUSTOMERS)

print(f"Total customers needed:      {total_needed:,}")
print(f"Daily eligible customers:    {DAILY_ELIGIBLE_CUSTOMERS:,}")
print(f"Estimated experiment length: {duration_days:.0f} days (~{duration_days/7:.1f} weeks)")


## 3. Sample size sensitivity: MDE, variance, power, confidence level

In [ ]:
mde_range = np.linspace(0.02, 0.15, 25)   # 2% to 15% relative lift
sample_sizes_by_mde = [
    required_sample_size(cohens_d(baseline_mean * m, baseline_std), alpha=ALPHA, power=POWER)
    for m in mde_range
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mde_range * 100, sample_sizes_by_mde, marker="o", color="#4C72B0")
ax.axvline(5, color="gray", linestyle="--", label="Our chosen MDE (5%)")
ax.set_xlabel("Minimum Detectable Effect (relative %)")
ax.set_ylabel("Required sample size per group")
ax.set_title("Sample size vs. MDE")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
std_multipliers = np.linspace(0.5, 2.0, 25)
sample_sizes_by_std = [
    required_sample_size(cohens_d(baseline_mean * MDE_RELATIVE, baseline_std * m), alpha=ALPHA, power=POWER)
    for m in std_multipliers
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(std_multipliers * baseline_std, sample_sizes_by_std, marker="o", color="#C44E52")
ax.axvline(baseline_std, color="gray", linestyle="--", label="Our observed std")
ax.set_xlabel("Revenue std. deviation ($)")
ax.set_ylabel("Required sample size per group")
ax.set_title("Sample size vs. metric variance\n(noisier metric -> more customers needed)")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
power_range = np.linspace(0.5, 0.99, 25)
sample_sizes_by_power = [
    required_sample_size(result.effect_size, alpha=ALPHA, power=p)
    for p in power_range
]

confidence_levels = [0.90, 0.95, 0.99]
fig, ax = plt.subplots(figsize=(8, 5))
for conf in confidence_levels:
    alpha_c = 1 - conf
    sizes = [required_sample_size(result.effect_size, alpha=alpha_c, power=p) for p in power_range]
    ax.plot(power_range, sizes, marker="o", label=f"{int(conf*100)}% confidence")

ax.set_xlabel("Desired power")
ax.set_ylabel("Required sample size per group")
ax.set_title("Sample size vs. power, by confidence level")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Power curve: what power do we actually get at different sample sizes?

In [ ]:
n_range = np.linspace(1000, 50000, 40).astype(int)
achieved_power = [power_given_n(result.effect_size, n, alpha=ALPHA) for n in n_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_range, achieved_power, color="#55A868")
ax.axhline(0.8, color="gray", linestyle="--", label="80% power target")
ax.axvline(result.required_n_per_group, color="gray", linestyle=":", label="Planned n/group")
ax.set_xlabel("Sample size per group")
ax.set_ylabel("Statistical power")
ax.set_title("Power curve at MDE = 5%")
ax.legend()
plt.tight_layout()
plt.show()


## Takeaways

- At our chosen MDE (5%), alpha (0.05) and power (80%), we need a fairly large sample per group
  (see the printed value above) — this is the direct consequence of revenue being a noisy,
  zero-inflated metric with a small standardized effect size.
- Given daily eligible traffic, that translates into several weeks of experiment runtime (see above).
- The sample-size curves show the two biggest levers if we need to move faster: **raise the MDE**
  (only try to detect bigger effects) or **reduce variance** (e.g. metric transformations,
  stratification, or CUPED-style pre-experiment covariate adjustment).

## Next: `04_ab_testing.ipynb` — actually run the experiment and reach a decision.